In [1]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import json
import os
import time
import boto3
from pathlib import Path
from datetime import datetime, timezone

In [2]:
print(f"Pandas: {pd.__version__}")
print(f"PyArrow: {pa.__version__}")

Pandas: 3.0.3
PyArrow: 24.0.0


In [3]:
s3_client = boto3.client(
        "s3",
        endpoint_url = "http://localhost:9000",
        aws_access_key_id = "minioadmin",
        aws_secret_access_key = "minioadmin123",
        region_name = "us-east-1"
    )

In [4]:
Path("../data/bronze/iris").mkdir(parents=True, exist_ok=True)
Path("../data/bronze/sintetico").mkdir(parents=True, exist_ok=True)
print("\u2705 Diretórios e conexão MinIO configurados!")

✅ Diretórios e conexão MinIO configurados!


In [5]:
URL_IRIS=("https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data")

In [6]:
COLUNAS_IRIS = [
    "comprimento_sepala_cm",
    "largura_sepala_cm",
    "comprimento_petala_cm",
    "largura_petala_cm",
    "especie",
]

In [7]:
df_iris = pd.read_csv(URL_IRIS, header = None, names = COLUNAS_IRIS)

In [8]:
df_iris["_fonte"] = "uci_iris"

In [9]:
df_iris["_ingerido_em"] = datetime.now(timezone.utc).isoformat()

In [10]:
df_iris["_versao_schema"]="1.0"

In [11]:
print(f"Shape do dataset: {df_iris.shape}")
print(f"\nTipos de dados: \n {df_iris.dtypes}")
print(f"\nPrimeiras 5 linhas:")
df_iris.head()

Shape do dataset: (150, 8)

Tipos de dados: 
 comprimento_sepala_cm    float64
largura_sepala_cm        float64
comprimento_petala_cm    float64
largura_petala_cm        float64
especie                      str
_fonte                       str
_ingerido_em                 str
_versao_schema               str
dtype: object

Primeiras 5 linhas:


,comprimento_sepala_cm,largura_sepala_cm,comprimento_petala_cm,largura_petala_cm,especie,_fonte,_ingerido_em,_versao_schema
0,5.1,3.5,1.4,0.2,Iris-setosa,uci_iris,2026-05-14T17:12:10.576035+00:00,1.0
1,4.9,3.0,1.4,0.2,Iris-setosa,uci_iris,2026-05-14T17:12:10.576035+00:00,1.0
2,4.7,3.2,1.3,0.2,Iris-setosa,uci_iris,2026-05-14T17:12:10.576035+00:00,1.0
3,4.6,3.1,1.5,0.2,Iris-setosa,uci_iris,2026-05-14T17:12:10.576035+00:00,1.0
4,5.0,3.6,1.4,0.2,Iris-setosa,uci_iris,2026-05-14T17:12:10.576035+00:00,1.0


In [13]:
print("+++ Estatísticas Descritivas+++")
print(df_iris.describe())
print(f"\nDistribuição por espécie:")
print(df_iris["especie"].value_counts())

+++ Estatísticas Descritivas+++
       comprimento_sepala_cm  largura_sepala_cm  comprimento_petala_cm  \
count             150.000000         150.000000             150.000000   
mean                5.843333           3.054000               3.758667   
std                 0.828066           0.433594               1.764420   
min                 4.300000           2.000000               1.000000   
25%                 5.100000           2.800000               1.600000   
50%                 5.800000           3.000000               4.350000   
75%                 6.400000           3.300000               5.100000   
max                 7.900000           4.400000               6.900000   

       largura_petala_cm  
count         150.000000  
mean            1.198667  
std             0.763161  
min             0.100000  
25%             0.300000  
50%             1.300000  
75%             1.800000  
max             2.500000  

Distribuição por espécie:
especie
Iris-setosa        50
I

In [14]:
# Criação de um dataset sintético (1 milhão de registros)
print("Gerando dataset sintético de 1.000.000 de registros...")
inicio = time.time()
np.random.seed(42)
N = 1_000_000
CATEGORIAS = ["Eletrônicos", "Roupas", "Alimentos", "Livros", "Esportes"]
STATUS = ["concluido", "cancelado", "pendente", "reembolsado"]
REGIOES = ["Sudeste", "Sul", "Nordeste", "Norte", "Centro-Oeste"]

Gerando dataset sintético de 1.000.000 de registros...


In [19]:
df_sintetico = pd.DataFrame({
    "id_pedido": range(1, N+1),
    "id_cliente": np.random.randint(1, 100_001, N),
    "id_produto": np.random.randint(1, 10_001, N),
    "categoria": np.random.choice(CATEGORIAS, N),
    "valor_unitario": np.round(np.random.uniform(5.0, 2000.0, N),2),
    "quantidade": np.random.randint(1,11,N),
    "status_pedido": np.random.choice(STATUS, N, p=[0.75, 0.10, 0.10, 0.05]),
    "regiao": np.random.choice(REGIOES, N),
    "data_pedido": pd.date_range(start="2022-01-01", periods=N, freq = "30s"),
    "avaliacao_cliente": np.random.choice([1,2,3,4,5,None], N, p=[0.05, 0.08, 0.15, 0.30, 0.37, 0.05]),
    "_fonte": "sistema_ecommerce_v2",
    "_ingerido_em": datetime.now(timezone.utc).isoformat(),
})

In [21]:
df_sintetico["valor_total"] = (df_sintetico["valor_unitario"] * df_sintetico["quantidade"]).round(2)

In [23]:
duracao = time.time() - inicio

In [24]:
print(f"Dataset gerado em {duracao:.2f}s")

Dataset gerado em 473.91s


In [25]:
BASE_PATH = Path("../data/bronze/sintetico")
inicio = time.time()
caminho_csv = BASE_PATH / "pedidos.csv"
df_sintetico.to_csv(caminho_csv, index=False)
tempo_escrita_csv = time.time() - inicio
tamanho_csv = caminho_csv.stat().st_size